In [0]:
%pip install sqlalchemy psycopg2-binary pandas
dbutils.library.restartPython()

In [0]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine

POSTGRES_HOST = "ep-summer-tooth-a8hwz3t3-pooler.eastus2.azure.neon.tech"
POSTGRES_PORT = "5432"
POSTGRES_DB = "Demo-Database"
POSTGRES_USER = "neondb_owner"
POSTGRES_PASSWORD = "npg_QU4WcsN2EXSi"

POSTGRES_URL = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{quote_plus(POSTGRES_PASSWORD)}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}?sslmode=require"
)

engine = create_engine(POSTGRES_URL)

In [0]:
# Importamos 'declarative_base', que es una fábrica para crear la clase base de nuestros modelos
from sqlalchemy.orm import declarative_base
# Importamos los tipos de datos y herramientas estructurales para definir las columnas
from sqlalchemy import Column, Integer, String, DateTime, Float, Text
# Importamos 'func' para poder usar funciones nativas de la base de datos (como obtener la hora actual)
from sqlalchemy.sql import func

# Creamos la clase 'Base'. Todas las tablas que definamos de ahora en adelante 
# deben heredar de esta clase para que SQLAlchemy las reconozca y las agrupe.
Base = declarative_base()

# --- PRIMERA TABLA: source_loads ---
class SourceLoad(Base):
    # __tablename__ es obligatorio: le dice a SQLAlchemy el nombre real que tendrá la tabla en PostgreSQL
    __tablename__ = "source_loads"

    # Definimos la clave primaria (Primary Key). Al ser Integer y primary_key=True, 
    # SQLAlchemy le dirá a PostgreSQL que sea autoincremental (tipo SERIAL).
    id = Column(Integer, primary_key=True)
    
    # Columnas de texto con un límite de caracteres. 'nullable=False' significa que NO aceptan valores nulos (NOT NULL)
    source_name = Column(String(100), nullable=False)
    source_type = Column(String(50), nullable=False)
    
    # Columna entera. 'default=0' hace que si no le pasamos un valor desde Python, ponga un 0.
    record_count = Column(Integer, default=0)
    status = Column(String(30), default="pending")
    
    # 'server_default=func.now()' es muy útil. Le dice a PostgreSQL que, al momento de insertar
    # un registro, calcule e inserte automáticamente la fecha y hora actual del servidor.
    created_at = Column(DateTime, server_default=func.now())


# --- SEGUNDA TABLA: product_snapshots ---
class ProductSnapshot(Base):
    __tablename__ = "product_snapshots"

    id = Column(Integer, primary_key=True)
    source = Column(String(50), nullable=False)
    product_name = Column(String(200), nullable=False)
    
    # Float para números con decimales (ideal para precios)
    price = Column(Float)
    currency = Column(String(10))
    
    # Text se usa para cadenas de texto muy largas o sin límite definido.
    # Aquí parece que se guardará un JSON crudo (raw_payload).
    raw_payload = Column(Text)

In [0]:
# Base.metadata es una colección (registro) de todas las tablas que definiste.
# .create_all(engine) le ordena a SQLAlchemy emitir los comandos CREATE TABLE
# en la base de datos conectada a través del motor (engine).
Base.metadata.create_all(engine)
print("Tablas creadas en Neon")

In [0]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
session = Session()

load_1 = SourceLoad(
    source_name="manual_test",
    source_type="setup",
    record_count=1,
    status="loaded"
)

product_1 = ProductSnapshot(
    source="manual_test",
    product_name="Producto de prueba",
    price=19.99,
    currency="USD",
    raw_payload='{"origen":"prueba","detalle":"registro inicial"}'
)

session.add(load_1)
session.add(product_1)
session.commit()

print("Datos insertados correctamente")

In [0]:
from sqlalchemy import text

with engine.connect() as conn:
    print("Tabla source_loads")
    result = conn.execute(text("SELECT * FROM source_loads ORDER BY id DESC LIMIT 5"))
    for row in result:
        print(row)

    print("\nTabla product_snapshots")
    result = conn.execute(text("SELECT * FROM product_snapshots ORDER BY id DESC LIMIT 5"))
    for row in result:
        print(row)